# Experiment 8: Dashboard, Responsible AI Reporting & Final Portfolio

**Course:** Applied Data Science (ADS)
**System:** Customer Support Emotion & Urgency Triage Microservice
**Artifacts:** Streamlit Dashboard, FastAPI Service, LightGBM Emotion Classifier, Responsible AI Report

---

## 1. Objective & System Architecture
This notebook demonstrates the end-to-end operational pipeline for the final Experiment 8 portfolio project:
1. **Model Loading & Scoring:** LightGBM + Negation-Aware TF-IDF + VADER Sentiment.
2. **Explainability (XAI / SHAP):** Token-level feature attribution.
3. **Responsible AI & Governance:** PII redaction and critical safety overrides.
4. **MLOps & Monitoring:** Population Stability Index (PSI) drift tracking against baseline.
5. **Operational Triage:** Dynamic SLA deadline assignment and REST API readiness.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from model_engine import EmotionEngine, get_engine
from pii import mask_pii
from ticket_store import TicketStore
from triage_service import TriageService

engine = get_engine()
print("Engine pipeline ready:", engine.pipeline is not None)

## 2. Real-Time Emotion & Sentiment Inference
Testing inference on representative customer support utterances.

In [ ]:
samples = [
    "My account was compromised and unauthorized charges were made!",
    "My package arrived completely crushed and two weeks late. Terrible service.",
    "Can you tell me how to renew my subscription before next month?",
    "Thank you Alex from support! You resolved my problem in minutes!"
]

results = []
for text in samples:
    pred = engine.predict(text)
    sentiment = engine.get_sentiment(text)
    results.append({
        "Message": text,
        "Predicted Emotion": pred["primary_emotion"],
        "Confidence": f"{pred['confidence']:.1%}",
        "Sentiment": sentiment["label"],
        "Compound Score": sentiment["compound"]
    })

pd.DataFrame(results)

## 3. Explainability & SHAP-Style Token Feature Attribution
Local model explainability using leave-one-token-out perturbation sensitivity to identify words driving predictions.

In [ ]:
test_message = "My account was hacked and I demand an immediate refund!"
attributions = engine.explain_prediction(test_message)

df_attr = pd.DataFrame(attributions)
print(f"Target Emotion: {engine.predict(test_message)['primary_emotion']}")
df_attr.head(8)

## 4. Responsible AI: PII Sanitization & Safety Overrides
Verifying that customer credit cards, emails, and phone numbers are scrubbed before persistence.

In [ ]:
raw_input = "Please refund my Visa 4111-2222-3333-4444 and call me at 555-839-2019 or email john.doe@acme.org"
sanitized = mask_pii(raw_input)
print("Raw Message:      ", raw_input)
print("Sanitized Message:", sanitized)
assert "4111" not in sanitized
assert "john.doe" not in sanitized

## 5. MLOps: Data Drift Monitoring (Population Stability Index / PSI)
Continuous statistical monitoring comparing recent text length and sentiment distributions against a 5,000-message baseline.

In [ ]:
drift_report = engine.check_data_drift(samples * 20)
print("Drift Status:       ", drift_report["status"])
print("Overall PSI:        ", drift_report["psi_score"])
print("Length PSI:         ", drift_report["feature_psi"]["length"])
print("Sentiment PSI:      ", drift_report["feature_psi"]["sentiment"])
print("Recommendation:     ", drift_report["recommendation"])

## 6. End-to-End Triage Service & Dynamic SLA Calculation

In [ ]:
store = TicketStore(database_url="sqlite:///:memory:")
service = TriageService(store, engine)

ticket = service.create_ticket(
    message="I am locked out of my corporate account and have a board presentation in 30 minutes!",
    external_id="PORTFOLIO-101",
    source="notebook",
    actor="lead_data_scientist"
)

print("Ticket ID:        ", ticket["id"])
print("Assigned Urgency: ", ticket["urgency"])
print("Primary Emotion:  ", ticket["primary_emotion"])
print("SLA Deadline:     ", ticket["sla_due_at"])
print("Routing Advice:   ", ticket["routing_recommendation"])